# Pipewise 功能测试

逐项验证 `pipewise` 2.0.0 的功能：先构造仿真订单数据（含 list / dict / ndarray / 类实例等非常规列），再按功能逐个断言。

**如何运行**：从项目根目录打开本 notebook，选择已安装 pandas 的 kernel，然后 *Run All*。第一个单元格会自动把项目根目录加入 `sys.path`，确保测试的是本地源码。

## 0. 环境自检

确保导入的是**本仓库的源码**（而不是 site-packages 里已安装的旧版本），
并打印依赖版本，方便排查环境问题。


In [1]:
import logging
import sys
import warnings
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "pipewise" / "__init__.py"
        ).is_file():
            return candidate
    raise RuntimeError(f"从 {start} 向上找不到 pipewise 项目根目录")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

import pipewise
from pipewise import Pipewise
from pipewise.errors import (
    PipewiseExecutionError,
    PipewiseInputColumnError,
    PipewiseInputSchemaError,
    PipewiseOutputAssignmentError,
    PipewiseOutputSchemaError,
    PipewiseRegistrationError,
    PipewiseTaskSelectionError,
)

assert Path(pipewise.__file__).resolve().is_relative_to(PROJECT_ROOT), (
    f"导入的不是本地源码: {pipewise.__file__}"
)

print("项目根目录 :", PROJECT_ROOT)
print("pipewise   :", Path(pipewise.__file__).resolve().relative_to(PROJECT_ROOT))
print("版本       :", pipewise.__version__)
print("pandas     :", pd.__version__, "| numpy:", np.__version__)

# 关闭 tqdm 进度条，让 notebook 输出聚焦在断言结果上。
# 想观察进度条效果，注释掉下面这行即可。
from pipewise import core as _pw_core  # noqa: E402

_pw_core._tqdm = None


项目根目录 : /Users/xiaoyuyu/Documents/work/dev/pipewise
pipewise   : pipewise/__init__.py
版本       : 2.0.0
pandas     : 3.0.5 | numpy: 2.5.3


### 测试脚手架

`check()` 记录每项结果并即时打印；`check_raises()` 断言某个调用抛出指定异常，
可选校验 `__cause__`。最后的汇总单元格会在有失败时抛错，保证 notebook 执行失败可见。


In [2]:
PASSED: list = []
FAILED: list = []


def check(name: str, condition: bool, detail: str = "") -> None:
    if condition:
        PASSED.append(name)
        print(f"✅ {name}")
    else:
        FAILED.append(name)
        print(f"❌ {name} {detail}")


def check_raises(name, exc_type, fn, cause_type=None):
    try:
        fn()
    except Exception as exc:  # noqa: BLE001 - 测试需要捕获任意异常类型
        ok = isinstance(exc, exc_type) and (
            cause_type is None or isinstance(exc.__cause__, cause_type)
        )
        cause_name = type(exc.__cause__).__name__ if exc.__cause__ else "-"
        check(name, ok, f"（实际 {type(exc).__name__} / cause={cause_name}）")
        return exc
    check(name, False, "（未抛出异常）")
    return None


class _LogCapture(logging.Handler):
    """把指定级别及以上的日志消息收集起来，供断言使用。"""

    def __init__(self, level=logging.WARNING):
        super().__init__(level=level)
        self.messages: list = []

    def emit(self, record):
        self.messages.append(record.getMessage())


def capture_logs(logger_name: str, level=logging.WARNING):
    handler = _LogCapture(level=level)
    logger = logging.getLogger(logger_name)
    logger.addHandler(handler)
    return handler, logger


## 1. 仿真数据

一份订单表，覆盖常规列（字符串 / 整数 / 浮点 / 时间）与**非常规单元格**：

| 列 | 类型 | 用途 |
|---|---|---|
| `order_id` / `customer` / `region` | 字符串 | 基础列、分组键 |
| `quantity` / `unit_price` / `discount` | 数值 | 向量化计算 |
| `created_at` | datetime | 时间类型 |
| `tags` | `list[str]` | 列表列 |
| `meta` | `dict` | 字典列 |
| `vec` | `np.ndarray` | ndarray 列 |
| `labels` | `str`（逗号分隔） | split → 列表列的输入 |

固定随机种子，保证每次运行结果一致。


In [3]:
rng = np.random.default_rng(20260912)
N = 15

orders = pd.DataFrame(
    {
        "order_id": [f"O{1000 + i}" for i in range(N)],
        "customer": rng.choice(["alice", "bob", "carol", "dave"], size=N),
        "region": rng.choice(["north", "south", "east", "west"], size=N),
        "quantity": rng.integers(1, 20, size=N),
        "unit_price": np.round(rng.uniform(5, 120, size=N), 2),
        "discount": np.round(rng.uniform(0, 0.3, size=N), 3),
        "created_at": pd.Timestamp("2026-01-01")
        + pd.to_timedelta(rng.integers(0, 120, size=N), unit="D"),
        "tags": [
            sorted(
                rng.choice(
                    ["urgent", "gift", "bulk", "fragile"],
                    size=int(rng.integers(0, 3)),
                    replace=False,
                ).tolist()
            )
            for _ in range(N)
        ],
        "meta": [
            {"channel": channel, "vip": bool(vip)}
            for channel, vip in zip(
                rng.choice(["web", "app"], size=N), rng.integers(0, 2, size=N)
            )
        ],
        "vec": [
            np.round(rng.normal(size=int(rng.integers(2, 5))), 3) for _ in range(N)
        ],
        # 追加在最后：rng 是顺序消费的，放在末尾不会改变上面各列的取值
        "labels": [
            ",".join(
                rng.choice(
                    ["urgent", "gift", "bulk", "fragile"],
                    size=int(rng.integers(1, 3)),
                    replace=False,
                ).tolist()
            )
            for _ in range(N)
        ],
    }
)

print(f"仿真数据：{orders.shape[0]} 行 × {orders.shape[1]} 列")
print("dtypes:", dict(orders.dtypes.astype(str)))
orders.head()


仿真数据：15 行 × 11 列
dtypes: {'order_id': 'str', 'customer': 'str', 'region': 'str', 'quantity': 'int64', 'unit_price': 'float64', 'discount': 'float64', 'created_at': 'datetime64[us]', 'tags': 'object', 'meta': 'object', 'vec': 'object', 'labels': 'str'}


,order_id,customer,region,quantity,unit_price,discount,created_at,tags,meta,vec,labels
0,O1000,dave,east,7,87.84,0.220,2026-01-06,[],"{'channel': 'app', 'vip': True}","[1.395, -0.38]",urgent
1,O1001,alice,north,7,94.25,0.251,2026-03-05,[],"{'channel': 'app', 'vip': True}","[-1.139, 1.744]",bulk
2,O1002,bob,south,14,119.35,0.081,2026-04-06,[bulk],"{'channel': 'web', 'vip': True}","[0.041, -0.084]","bulk,urgent"
3,O1003,dave,west,19,68.98,0.162,2026-04-13,[fragile],"{'channel': 'web', 'vip': True}","[-0.151, 0.712, -2.369]","fragile,urgent"
4,O1004,bob,west,8,51.19,0.154,2026-04-11,[],"{'channel': 'web', 'vip': False}","[-0.289, 0.983, 0.11]",bulk


## 2. 基础注册与向量化多列输出

`register(outputs=[...])` 注册多列输出；带该函数参数的列名自动映射为输入。
默认 `vectorized=True`，整列 `Series` 传入。同时在最后断言「默认不修改原表」。


In [4]:
pw = Pipewise(orders)


@pw.register(outputs=["gross", "net"])
def amount(quantity, unit_price, discount):
    gross = quantity * unit_price
    return gross, gross * (1 - discount)


result = pw.run()
expected_gross = orders["quantity"] * orders["unit_price"]

check("向量化：多列输出 gross 正确", np.allclose(result["gross"], expected_gross))
check(
    "向量化：多列输出 net 正确",
    np.allclose(result["net"], expected_gross * (1 - orders["discount"])),
)
check("默认不修改原 DataFrame", "gross" not in orders.columns)
check("新列追加在末尾", list(result.columns)[-2:] == ["gross", "net"])
result[["order_id", "gross", "net"]].head()


✅ 向量化：多列输出 gross 正确
✅ 向量化：多列输出 net 正确
✅ 默认不修改原 DataFrame
✅ 新列追加在末尾


,order_id,gross,net
0,O1000,614.88,479.60640
1,O1001,659.75,494.15275
2,O1002,1670.90,1535.55710
3,O1003,1310.62,1098.29956
4,O1004,409.52,346.45392


## 3. 单列输出与「仅副作用」模式

`outputs="col"` 写回单列；`outputs=None` 表示只执行副作用、不写回任何列。


In [5]:
pw = Pipewise(orders)
side_effects = []


@pw.register(outputs="is_high_value")
def high_value(quantity, unit_price):
    return quantity * unit_price > 500


@pw.register(outputs=None)
def record_total(quantity):
    side_effects.append(int(quantity.sum()))


result = pw.run()

expected_flag = (orders["quantity"] * orders["unit_price"] > 500).tolist()
check("单列输出：布尔列正确", result["is_high_value"].tolist() == expected_flag)
check(
    "outputs=None：不写回任何列",
    set(result.columns) == set(orders.columns) | {"is_high_value"},
)
check("outputs=None：副作用被执行", side_effects == [int(orders["quantity"].sum())])


✅ 单列输出：布尔列正确
✅ outputs=None：不写回任何列
✅ outputs=None：副作用被执行


## 4. 带类型声明的输出

`outputs={"col": type}` 在写回后自动 `astype`，适合把计算结果固定成目标 dtype。


In [6]:
pw = Pipewise(orders)


@pw.register(outputs={"total": float, "is_bulk": bool})
def amount_typed(quantity, unit_price):
    return quantity * unit_price, quantity >= 10


result = pw.run()

check("类型声明：float 转换生效", str(result["total"].dtype).startswith("float"))
check("类型声明：bool 转换生效", str(result["is_bulk"].dtype) == "bool")
check(
    "类型声明：数值计算正确",
    np.allclose(result["total"], orders["quantity"] * orders["unit_price"]),
)


✅ 类型声明：float 转换生效
✅ 类型声明：bool 转换生效
✅ 类型声明：数值计算正确


## 5. 动态字典输出

`outputs="dict"` 允许每一行按条件返回不同的列。这里用逐行模式，让「有折扣才写
`line_discounted`」的分支成立，并验证缺失行是 NaN 而非错误值。


In [7]:
pw = Pipewise(orders)


@pw.register(outputs="dict", vectorized=False)
def enrich(quantity, unit_price, discount):
    total = round(float(quantity * unit_price), 2)
    row = {"line_total": total}
    if discount > 0:
        row["line_discounted"] = round(total * (1 - discount), 2)
    return row


result = pw.run()
discounted_rows = int((orders["discount"] > 0).sum())

check(
    "dict 输出：按行生成列",
    {"line_total", "line_discounted"} <= set(result.columns),
)
check(
    "dict 输出：条件列只在需要的行有值",
    int(result["line_discounted"].notna().sum()) == discounted_rows,
)
check(
    "dict 输出：line_total 正确",
    np.allclose(
        result["line_total"], np.round(orders["quantity"] * orders["unit_price"], 2)
    ),
)
result[["order_id", "line_total", "line_discounted"]].head()


✅ dict 输出：按行生成列
✅ dict 输出：条件列只在需要的行有值
✅ dict 输出：line_total 正确


,order_id,line_total,line_discounted
0,O1000,614.88,479.61
1,O1001,659.75,494.15
2,O1002,1670.90,1535.56
3,O1003,1310.62,1098.30
4,O1004,409.52,346.45


## 6. 逐行执行

`vectorized=False` 时每个分组/每行的值以 **Python 标量** 传入，可以直接写
`if quantity >= 10:` 这类分支逻辑。


In [8]:
pw = Pipewise(orders)


@pw.register(outputs="size_class", vectorized=False)
def classify_row(quantity, unit_price):
    if quantity >= 10 and unit_price < 50:
        return "bulk-cheap"
    if quantity >= 10:
        return "bulk-pricey"
    return "small"


def expected_class(quantity, unit_price):
    if quantity >= 10 and unit_price < 50:
        return "bulk-cheap"
    if quantity >= 10:
        return "bulk-pricey"
    return "small"


result = pw.run()
expected = [
    expected_class(q, p)
    for q, p in zip(orders["quantity"], orders["unit_price"])
]

check("逐行执行：结果与逐行语义一致", result["size_class"].tolist() == expected)
check("逐行执行：行数不变", len(result) == len(orders))


✅ 逐行执行：结果与逐行语义一致
✅ 逐行执行：行数不变


## 7. 向量化 → 逐行 回退（需显式开启）

2.0 起回退**默认关闭**。开启后，向量化调用若因 Series 不兼容而失败，会改为逐行重跑，
并发出 `RuntimeWarning`。


In [9]:
pw = Pipewise(orders)


@pw.register(outputs="branch_label", fallback_on_vectorized_error=True)
def branch_on_quantity(quantity):
    if quantity >= 10:  # 整列比较会抛 ValueError -> 触发回退
        return "big"
    return "small"


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = pw.run()

fell_back = any("fell back" in str(w.message) for w in caught)
expected = ["big" if q >= 10 else "small" for q in orders["quantity"]]

check("回退(opt-in)：结果正确", result["branch_label"].tolist() == expected)
check("回退(opt-in)：发出 RuntimeWarning 提示", fell_back)


Function 'branch_on_quantity' may be vectorized-incompatible: `quantity >= 10` is used as a branch condition — evaluating a Series for truth raises ValueError (`truth value of a Series is ambiguous`). Use `vectorized=False` to branch per row, or a vectorized expression such as `numpy.where` / `Series.where`.


✅ 回退(opt-in)：结果正确
✅ 回退(opt-in)：发出 RuntimeWarning 提示


### 默认行为：不回退，直接报错

同一个函数不开启回退时，应向用户暴露失败，并提示改用 `vectorized=False`。
这样避免「已经执行过一次的函数体被再次执行」导致副作用重复。


In [10]:
pw = Pipewise(orders)


@pw.register(outputs="branch_label")
def branch_on_quantity(quantity):
    if quantity >= 10:
        return "big"
    return "small"


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    exc = check_raises(
        "默认不回退：抛出 PipewiseExecutionError",
        PipewiseExecutionError,
        pw.run,
    )

check(
    "默认不回退：保留原始异常上下文",
    exc is not None and isinstance(exc.__cause__, ValueError),
)
check(
    "默认不回退：提示改用 vectorized=False",
    any("vectorized=False" in str(w.message) for w in caught),
)


Function 'branch_on_quantity' may be vectorized-incompatible: `quantity >= 10` is used as a branch condition — evaluating a Series for truth raises ValueError (`truth value of a Series is ambiguous`). Use `vectorized=False` to branch per row, or a vectorized expression such as `numpy.where` / `Series.where`.
Function 'branch_on_quantity' raised ValueError during vectorized execution: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().. If it is meant to run per row, register it with vectorized=False (or pass fallback_on_vectorized_error=True).


✅ 默认不回退：抛出 PipewiseExecutionError
✅ 默认不回退：保留原始异常上下文
✅ 默认不回退：提示改用 vectorized=False


## 8. GroupBy 分组执行

`groupby="col"` 或 `groupby=["c1","c2"]`：按分组 split-apply-combine，组内独立计算后
再回写原表。


In [11]:
pw = Pipewise(orders)


@pw.register(outputs="region_share", groupby="region")
def share_within_region(quantity, unit_price):
    total = quantity * unit_price
    return total / total.sum()


result = pw.run()
per_region = result.groupby(orders["region"])["region_share"].sum()

check("GroupBy：组内占比求和为 1", np.allclose(per_region.values, 1.0))
check("GroupBy：行数不变", len(result) == len(orders))

pw_multi = Pipewise(orders)


@pw_multi.register(outputs="echo", groupby=["region", "customer"], vectorized=False)
def echo_quantity(quantity):
    return quantity


result_multi = pw_multi.run()
check(
    "GroupBy：多列分组 + 逐行执行可用",
    result_multi["echo"].tolist() == orders["quantity"].tolist(),
)


✅ GroupBy：组内占比求和为 1
✅ GroupBy：行数不变
✅ GroupBy：多列分组 + 逐行执行可用


## 9. Schema 校验

支持 `dtype` / `nullable` / `allowed_values` / `min` / `max`，可声明在**管道输入**、
**任务输入**、**任务输出**三处。这里逐条验证通过路径与拒绝路径。


In [12]:
# 1) 合法输入通过
ok = Pipewise(
    orders,
    input_schema={"quantity": {"dtype": "integer", "nullable": False, "min": 1}},
)
check("Schema：合法输入通过", ok.run() is not None)

# 2) 缺失列
missing = Pipewise(orders, input_schema={"not_a_column": {"dtype": "number"}})
check_raises(
    "Schema：声明列缺失时报错", PipewiseInputSchemaError, missing.run
)

# 3) 非空约束被违反
with_null = orders.copy()
with_null.loc[0, "unit_price"] = None
nulls = Pipewise(with_null, input_schema={"unit_price": {"nullable": False}})
check_raises(
    "Schema：nullable=False 被违反时报错", PipewiseInputSchemaError, nulls.run
)

# 4) min 越界
too_low = Pipewise(orders, input_schema={"quantity": {"min": 1000}})
check_raises("Schema：min 越界时报错", PipewiseInputSchemaError, too_low.run)

# 5) allowed_values 越界
restricted = Pipewise(
    orders, input_schema={"region": {"allowed_values": ["north", "south"]}}
)
check_raises(
    "Schema：allowed_values 越界时报错", PipewiseInputSchemaError, restricted.run
)

# 6) 任务输出 schema 越界
pw = Pipewise(orders)


@pw.register(outputs="score", output_schema={"score": {"dtype": "integer", "max": 100}})
def score(quantity):
    return quantity * 100


check_raises(
    "Schema：任务输出越界时报错",
    PipewiseExecutionError,
    pw.run,
    cause_type=PipewiseOutputSchemaError,
)


✅ Schema：合法输入通过
✅ Schema：声明列缺失时报错
✅ Schema：nullable=False 被违反时报错
✅ Schema：min 越界时报错
✅ Schema：allowed_values 越界时报错
✅ Schema：任务输出越界时报错


pipewise.errors.PipewiseExecutionError("Task 'score' failed, all changes rolled back.")

## 10. 失败回滚

任一任务失败时，本次执行期间的**所有列改动**都要被撤销，原始异常保留在 `__cause__`。
这里用 `inplace=True` 让改动落在内部表上，以便验证回滚效果。


In [13]:
pw = Pipewise(orders.copy())


@pw.register(outputs="temporary")
def ok_step(quantity):
    return quantity * 2


@pw.register(outputs="boom")
def failing_step(quantity):
    raise RuntimeError("模拟下游任务失败")


columns_before = list(pw.data.columns)
quantity_before = pw.data["quantity"].tolist()

exc = check_raises(
    "回滚：失败时抛 PipewiseExecutionError",
    PipewiseExecutionError,
    lambda: pw.run(inplace=True),
)

check(
    "回滚：上下文是原始异常",
    exc is not None and isinstance(exc.__cause__, RuntimeError),
)
check("回滚：新增列被撤销", list(pw.data.columns) == columns_before)
check("回滚：原有数据未被污染", pw.data["quantity"].tolist() == quantity_before)


✅ 回滚：失败时抛 PipewiseExecutionError
✅ 回滚：上下文是原始异常
✅ 回滚：新增列被撤销
✅ 回滚：原有数据未被污染


## 11. 任务管理

`tasks` 查看摘要、`plan()` 打印执行计划（走 logging）、`run(task=...)` 只跑单个任务、
`remove()` / `clear()` 增删任务。


In [14]:
pw = Pipewise(orders)


@pw.register(outputs="ordinal")
def step_ordinal(order_id):
    return [int(value[1:]) for value in order_id]


@pw.register(outputs="doubled")
def step_doubled(quantity):
    return quantity * 2


check(
    "tasks：摘要字段正确",
    [tuple(entry) for entry in pw.tasks]
    == [("step_ordinal", ["ordinal"], None, True), ("step_doubled", ["doubled"], None, True)],
)

handler, logger = capture_logs("pipewise.core", level=logging.INFO)
logger.setLevel(logging.INFO)
pw.plan()
logger.removeHandler(handler)
plan_text = "\n".join(handler.messages)
check(
    "plan()：包含执行计划与任务名",
    "Execution Plan:" in plan_text
    and "step_ordinal" in plan_text
    and "step_doubled" in plan_text,
)

single = pw.run(task="step_ordinal")
check(
    "run(task=)：只执行指定任务",
    set(single.columns) == set(orders.columns) | {"ordinal"},
)
check(
    "run(task=)：结果正确",
    single["ordinal"].tolist() == [int(value[1:]) for value in orders["order_id"]],
)

check_raises(
    "run(task=)：任务名不存在时报错",
    PipewiseTaskSelectionError,
    lambda: pw.run(task="no_such_task"),
)

check("remove()：按引用删除任务", pw.remove(step_doubled) is True)
pw.clear()
check("clear()：清空所有任务", pw.tasks == [])


✅ tasks：摘要字段正确
✅ plan()：包含执行计划与任务名
✅ run(task=)：只执行指定任务
✅ run(task=)：结果正确
✅ run(task=)：任务名不存在时报错
✅ remove()：按引用删除任务
✅ clear()：清空所有任务


## 12. 注册期与输入列的校验

非法 `outputs`、`output_schema` 引用未声明的输出列、以及缺失的输入列，都应在
执行前/执行时报出明确的 Pipewise 异常。


In [15]:
pw = Pipewise(orders)

check_raises(
    "注册校验：非法 outputs 报 PipewiseRegistrationError",
    PipewiseRegistrationError,
    lambda: pw.register(outputs=3.14)(lambda quantity: quantity),
)

check_raises(
    "注册校验：output_schema 引用未声明列时报错",
    PipewiseRegistrationError,
    lambda: pw.register(outputs="b", output_schema={"c": {"dtype": "integer"}})(
        lambda quantity: quantity
    ),
)

pw_missing = Pipewise(orders)


@pw_missing.register(outputs="x")
def needs_missing_column(not_a_column):
    return not_a_column


check_raises(
    "输入列校验：输入列缺失时报错",
    PipewiseExecutionError,
    pw_missing.run,
    cause_type=PipewiseInputColumnError,
)


✅ 注册校验：非法 outputs 报 PipewiseRegistrationError
✅ 注册校验：output_schema 引用未声明列时报错
✅ 输入列校验：输入列缺失时报错


pipewise.errors.PipewiseExecutionError("Task 'needs_missing_column' failed, all changes rolled back.")

## 13. 索引对齐

向量化函数返回 `Series` 时，按**标签**对齐到目标索引，避免乱序返回导致数据错位。


In [16]:
custom = pd.DataFrame({"a": [1, 2, 3]}, index=[10, 20, 30])
pw = Pipewise(custom)


@pw.register(outputs="aligned")
def reverse_index(a):
    return pd.Series([100, 200, 300], index=[30, 20, 10])


result = pw.run()

check(
    "索引对齐：乱序返回的 Series 按标签对齐",
    result["aligned"].tolist() == [300, 200, 100],
)
check("索引对齐：保留原索引", result.index.tolist() == [10, 20, 30])


✅ 索引对齐：乱序返回的 Series 按标签对齐
✅ 索引对齐：保留原索引


## 14. 非常规数据结构的兼容性

列中可以放非标量值。下面验证 `list` / `dict` / `set` / `tuple` / `np.ndarray`
在向量化与逐行两种模式下的处理。


In [17]:
pw = Pipewise(orders)


@pw.register(outputs="tag_count")
def count_tags(tags):
    return tags.str.len()


@pw.register(outputs="tag_summary", vectorized=False)
def summarize_tags(tags):
    return "|".join(sorted(tags))


@pw.register(outputs="vec_len", vectorized=False)
def vec_len(vec):
    return len(vec)


@pw.register(outputs=["meta_channel", "meta_vip"], vectorized=False)
def unpack_meta(meta):
    return meta["channel"], meta["vip"]


result = pw.run()

check(
    "非常规：list 列（向量化 .str.len）",
    result["tag_count"].tolist() == [len(t) for t in orders["tags"]],
)
check(
    "非常规：list 列（逐行 join）",
    result["tag_summary"].tolist() == ["|".join(sorted(t)) for t in orders["tags"]],
)
check(
    "非常规：np.ndarray 列（逐行 len）",
    result["vec_len"].tolist() == [len(v) for v in orders["vec"]],
)
check(
    "非常规：dict 列（逐行解包为多列）",
    result["meta_channel"].tolist() == [m["channel"] for m in orders["meta"]],
)
check(
    "非常规：dict 列（布尔取值）",
    result["meta_vip"].tolist() == [m["vip"] for m in orders["meta"]],
)

# set / tuple / ndarray 的独立小样例
exotic = pd.DataFrame(
    {
        "tags": [{"a", "b"}, {"c"}],
        "vec": [np.array([1, 2, 3]), np.array([4, 5])],
        "pairs": [(1, 2), (3, 4)],
    }
)
pw_exotic = Pipewise(exotic)


@pw_exotic.register(outputs="set_size", vectorized=False)
def set_size(tags):
    return len(tags)


@pw_exotic.register(outputs="vec_sum", vectorized=False)
def vec_sum(vec):
    return float(vec.sum())


@pw_exotic.register(outputs="pair_total", vectorized=False)
def pair_total(pairs):
    return sum(pairs)


exotic_result = pw_exotic.run()
check("非常规：set 列", exotic_result["set_size"].tolist() == [2, 1])
check("非常规：ndarray 求和", exotic_result["vec_sum"].tolist() == [6.0, 9.0])
check("非常规：tuple 列", exotic_result["pair_total"].tolist() == [3, 7])


✅ 非常规：list 列（向量化 .str.len）
✅ 非常规：list 列（逐行 join）
✅ 非常规：np.ndarray 列（逐行 len）
✅ 非常规：dict 列（逐行解包为多列）
✅ 非常规：dict 列（布尔取值）
✅ 非常规：set 列
✅ 非常规：ndarray 求和
✅ 非常规：tuple 列


### 类对象与类实例

列里可以放自定义类的实例，逐行读取其属性；二维 `np.ndarray` 返回值可作为多列输出。


In [25]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"


points = pd.DataFrame({"pt": [Point(1, 2), Point(3, 4)]})
pw = Pipewise(points)


@pw.register(outputs=["x", "y"], vectorized=False)
def unpack_point(pt):
    return pt.x, pt.y


@pw.register(outputs="norm", vectorized=False)
def hypot(pt):
    return (pt.x**2 + pt.y**2) ** 0.5


result = pw.run()
check(
    "非常规：类实例属性解包为多列",
    (result["x"].tolist(), result["y"].tolist()) == ([1, 3], [2, 4]),
)
check(
    "非常规：类实例参与计算",
    np.allclose(result["norm"], [np.hypot(1, 2), np.hypot(3, 4)]),
)

# 二维 ndarray -> 多列输出
pw_2d = Pipewise(orders)


@pw_2d.register(outputs=["gross2", "net2"])
def amount_2d(quantity, unit_price, discount):
    gross = (quantity * unit_price).to_numpy()
    return np.column_stack([gross, gross * (1 - discount.to_numpy())])


result_2d = pw_2d.run()
check(
    "非常规：二维 ndarray 作为多列输出",
    np.allclose(result_2d["gross2"], orders["quantity"] * orders["unit_price"]),
)




✅ 非常规：类实例属性解包为多列
✅ 非常规：类实例参与计算
✅ 非常规：二维 ndarray 作为多列输出


### 字符串 split → 列表列

把一列用分隔符拼接的字符串拆成列表列，有三种写法，行为各不相同：

| 写法 | 代码 | 说明 |
|---|---|---|
| 向量化（推荐） | `s.str.split(",")` | 一次处理整列，返回列表列 |
| 逐行 | `vectorized=False` + `s.split(",")` | 每行传标量，直接调 `str.split` |
| 回退 | 默认写法 + `fallback_on_vectorized_error=True` | 先试向量化，失败后逐行重跑 |

注册时 AST 检测会对 `s.split(...)` 发出**危险告警** —— `Series` 没有 `.split` 方法。
而在默认（不回退）配置下，这种写法会在运行时报错。三种写法产出的结果应当完全一致。


In [26]:
orders.head(3)

,order_id,customer,region,quantity,unit_price,discount,created_at,tags,meta,vec,labels
0,O1000,dave,east,7,87.84,0.220,2026-01-06,[],"{'channel': 'app', 'vip': True}","[1.395, -0.38]",urgent
1,O1001,alice,north,7,94.25,0.251,2026-03-05,[],"{'channel': 'app', 'vip': True}","[-1.139, 1.744]",bulk
2,O1002,bob,south,14,119.35,0.081,2026-04-06,[bulk],"{'channel': 'web', 'vip': True}","[0.041, -0.084]","bulk,urgent"


In [ ]:
print("labels 列（逗号分隔的字符串）：", orders["labels"].head(3).tolist())

# --- 写法 1：向量化，用 .str.split（推荐）---
pw_vec = Pipewise(orders)

@pw_vec.register(outputs="labels_vec")
def split_vectorized(labels):
    return labels.str.split(",")


result_vec = pw_vec.run()

In [ ]:
check(
    "split：向量化 .str.split 得到列表列",
    result_vec["labels_vec"].tolist() == [s.split(",") for s in orders["labels"]],
)
check(
    "split：结果每个单元格都是 list",
    all(isinstance(value, list) for value in result_vec["labels_vec"]),
)

# --- 写法 2：逐行 vectorized=False ---
pw_row = Pipewise(orders)


@pw_row.register(outputs="labels_row", vectorized=False)
def split_rowwise(labels):
    return labels.split(",")


result_row = pw_row.run()
check(
    "split：逐行 s.split 与向量化结果一致",
    result_row["labels_row"].tolist() == result_vec["labels_vec"].tolist(),
)

# --- 写法 3：显式开启回退（向量化失败后自动逐行重跑）---
pw_fallback = Pipewise(orders)


@pw_fallback.register(outputs="labels_fallback", fallback_on_vectorized_error=True)
def split_with_fallback(labels):
    return labels.split(",")


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result_fallback = pw_fallback.run()

check(
    "split：开启回退后结果一致",
    result_fallback["labels_fallback"].tolist() == result_vec["labels_vec"].tolist(),
)
check(
    "split：回退时发出 RuntimeWarning",
    any("fell back" in str(w.message) for w in caught),
)

# --- 默认写法（不回退）应当直接报错 ---
pw_strict = Pipewise(orders)


@pw_strict.register(outputs="labels_strict")
def split_strict(labels):
    return labels.split(",")


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    check_raises(
        "split：默认不回退，抛出 PipewiseExecutionError",
        PipewiseExecutionError,
        pw_strict.run,
        cause_type=AttributeError,
    )

check(
    "split：报错提示改用 vectorized=False",
    any("vectorized=False" in str(w.message) for w in caught),
)

# --- 拆出来的列表列可以继续被后续任务处理 ---
pw_chain = Pipewise(orders)


@pw_chain.register(outputs="label_list")
def labels_to_list(labels):
    return labels.str.split(",")


@pw_chain.register(outputs="label_count", vectorized=False)
def count_labels(label_list):
    return len(label_list)


chained = pw_chain.run()
check(
    "split：列表列可被后续任务继续处理",
    chained["label_count"].tolist() == [len(s.split(",")) for s in orders["labels"]],
)
chained[["order_id", "labels", "label_list", "label_count"]].head()

Function 'split_with_fallback' may be vectorized-incompatible: `labels.split(',')` raises AttributeError on a Series — use `.str.split()` (vectorized) or `vectorized=False` (row-wise).
Function 'split_strict' may be vectorized-incompatible: `labels.split(',')` raises AttributeError on a Series — use `.str.split()` (vectorized) or `vectorized=False` (row-wise).
Function 'split_strict' raised AttributeError during vectorized execution: 'Series' object has no attribute 'split'. If it is meant to run per row, register it with vectorized=False (or pass fallback_on_vectorized_error=True).


labels 列（逗号分隔的字符串）： ['urgent', 'bulk', 'bulk,urgent']
✅ split：向量化 .str.split 得到列表列
✅ split：结果每个单元格都是 list
✅ split：逐行 s.split 与向量化结果一致
✅ split：开启回退后结果一致
✅ split：回退时发出 RuntimeWarning
✅ split：默认不回退，抛出 PipewiseExecutionError
✅ split：报错提示改用 vectorized=False
✅ split：列表列可被后续任务继续处理


,order_id,labels,label_list,label_count
0,O1000,urgent,[urgent],1
1,O1001,bulk,[bulk],1
2,O1002,"bulk,urgent","[bulk, urgent]",2
3,O1003,"fragile,urgent","[fragile, urgent]",2
4,O1004,bulk,[bulk],1


## 15. 输出形状守卫

输出长度/形状与表不匹配时，应抛出带函数名与列名的
`PipewiseOutputAssignmentError`（包在 `PipewiseExecutionError` 中），
而不是让 pandas 报晦涩错误，或静默广播单个值。


In [20]:
pw = Pipewise(orders)


@pw.register(outputs="bad_len")
def wrong_length(quantity):
    return [1]  # 只有 1 个值，表有 N 行


exc = check_raises(
    "形状守卫：长度不符时报错",
    PipewiseExecutionError,
    pw.run,
    cause_type=PipewiseOutputAssignmentError,
)
check(
    "形状守卫：错误信息含列名",
    exc is not None and "'bad_len'" in str(exc.__cause__),
)

# 参差的逐行多列返回
ragged = pd.DataFrame({"quantity": [3, 12]})
pw_ragged = Pipewise(ragged)


@pw_ragged.register(outputs=["p", "q"], vectorized=False)
def ragged_rows(quantity):
    if quantity >= 10:
        return quantity, quantity * 2
    return (quantity,)


exc = check_raises(
    "形状守卫：参差返回值时报错",
    PipewiseExecutionError,
    pw_ragged.run,
    cause_type=PipewiseOutputAssignmentError,
)
check(
    "形状守卫：参差错误定位到具体行",
    exc is not None and "row" in str(exc.__cause__),
)

# 标量广播仍然是合法用法
pw_broadcast = Pipewise(orders)


@pw_broadcast.register(outputs="constant")
def constant(quantity):
    return 0


check(
    "形状守卫：标量广播仍受支持",
    pw_broadcast.run()["constant"].tolist() == [0] * len(orders),
)


✅ 形状守卫：长度不符时报错
✅ 形状守卫：错误信息含列名
✅ 形状守卫：参差返回值时报错
✅ 形状守卫：参差错误定位到具体行
✅ 形状守卫：标量广播仍受支持


## 16. AST 向量化危险检测

注册时会扫描函数源码，对**真正的输入列参数**提示潜在不兼容写法；局部变量与辅助
对象不应误报。为了让 `inspect.getsource` 可用，把样例函数写到临时模块再导入。


In [21]:
import importlib.util
import tempfile
import textwrap

SAMPLES = "\n".join(
    [
        "def branch_on_column(a):",
        "    if a > 10:",
        '        return "big"',
        '    return "small"',
        "",
        "def uses_only_locals(a):",
        '    tag = "TAG"',
        "    parts = [1, 2, 3]",
        '    return a.str.upper() + tag.upper() + str(len(parts))',
        "",
    ]
)

sample_dir = Path(tempfile.mkdtemp(prefix="pipewise_hazard_"))
sample_path = sample_dir / "hazard_samples.py"
sample_path.write_text(SAMPLES)

spec = importlib.util.spec_from_file_location("pipewise_hazard_samples", sample_path)
samples = importlib.util.module_from_spec(spec)
spec.loader.exec_module(samples)

pw = Pipewise(orders)
handler, logger = capture_logs("pipewise._hazards")
pw.register(outputs="cls")(samples.branch_on_column)
pw.register(outputs="txt")(samples.uses_only_locals)
logger.removeHandler(handler)
messages = handler.messages

check(
    "AST 检测：列用于分支条件时报危险告警",
    any("branch condition" in m for m in messages),
)
check(
    "AST 检测：只使用局部变量时不误报",
    not any("uses_only_locals" in m for m in messages),
)
check("AST 检测：本例恰好命中 1 条", len(messages) == 1, f"（实际 {len(messages)}: {messages}）")


✅ AST 检测：列用于分支条件时报危险告警
✅ AST 检测：只使用局部变量时不误报
✅ AST 检测：本例恰好命中 1 条


### 默认值参数吞掉同名列的告警

参数带默认值时不从 DataFrame 读取。若存在同名输入列，注册时应给出告警，
避免「列被静默忽略」。


In [22]:
pw = Pipewise(orders)

handler, logger = capture_logs("pipewise._tasks")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    @pw.register(outputs="shadow_demo")
    def shadowed(quantity, region="north"):
        return quantity

logger.removeHandler(handler)

check(
    "默认值告警：同名输入列被忽略时告警",
    any("with a default value" in m for m in handler.messages),
)
check(
    "默认值告警：同时发出 RuntimeWarning",
    any("default value" in str(w.message) for w in caught),
)


✅ 默认值告警：同名输入列被忽略时告警
✅ 默认值告警：同时发出 RuntimeWarning


## 17. 汇总

所有断言结果在此汇总；若有失败项，本单元格会抛错，让 notebook 执行以失败告终。


In [23]:
print("=" * 60)
print(f"通过 {len(PASSED)} 项，失败 {len(FAILED)} 项")

if FAILED:
    print("失败项：")
    for name in FAILED:
        print("  -", name)
    raise AssertionError(f"{len(FAILED)} 项功能测试未通过")

print("全部 pipewise 功能测试通过 ✅")


通过 74 项，失败 0 项
全部 pipewise 功能测试通过 ✅
